# Visualize Graph Partitioning with METIS

This notebook loads the static graph structure from the provided data files, partitions the graph using METIS (as done in the Cluster-GCN preprocessing step), and visualizes the resulting clusters.

In [11]:
import torch
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch_geometric.data import Data
import geopandas as gpd # For reading coordinates from SHP file

# --- Import the necessary functions from data_utils --- 
# Ensure data_utils.py is in the same directory or accessible via PYTHONPATH
try:
    from utils.cluster_utils import load_base_graph_structure, partition_graph
except ImportError as e:
    print(f"Error importing from data_utils: {e}\nEnsure data_utils.py is accessible.")
    # Define dummy functions if import fails, so the rest of the notebook can be understood
    def load_base_graph_structure(*args, **kwargs):
        print("Dummy load_base_graph_structure called.")
        # Return a dummy Data object for demonstration
        num_nodes = 100
        edge_index = torch.randint(0, num_nodes, (2, num_nodes * 2), dtype=torch.long)
        x = torch.randn(num_nodes, 3)
        edge_attr = torch.randn(num_nodes * 2, 3)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, num_nodes=num_nodes)
    
    def partition_graph(data, num_clusters):
        print(f"Dummy partition_graph called for {num_clusters} clusters.")
        data.part = torch.randint(0, num_clusters, (data.num_nodes,))
        return data

RuntimeError: Could not locate METIS dll. Please set the METIS_DLL environment variable to its full path.

## 1. Configuration

Set the paths to your raw data files and the desired number of clusters.

In [4]:
# --- Configuration --- 
RAW_DIR = './data/raw' # ADJUST: Path to your raw data directory
NODES_SHP_FILE = 'your_nodes_file.shp' # ADJUST: Name of your nodes shapefile
EDGES_SHP_FILE = 'your_edges_file.shp' # ADJUST: Name of your edges shapefile
HEC_RAS_GEO_FILE = 'your_representative_geo.hdf' # ADJUST: Name of a HEC-RAS file for geometry

NUM_CLUSTERS = 50 # ADJUST: The number of clusters you want METIS to create

# Visualization parameters
FIGURE_SIZE = (12, 10)
NODE_MARKER_SIZE = 5
COLORMAP = 'viridis' # Colormap for clusters (e.g., 'viridis', 'jet', 'tab20')

## 2. Load Static Graph Structure

Load the node features, edge connectivity, and edge features needed to define the graph structure for partitioning.

In [5]:
print("Loading base graph structure...")
try:
    base_graph: Data = load_base_graph_structure(
        RAW_DIR,
        NODES_SHP_FILE,
        EDGES_SHP_FILE,
        HEC_RAS_GEO_FILE
    )
    print(f"Successfully loaded graph: {base_graph}")
except FileNotFoundError as e:
    print(f"\nError loading data: {e}")
    print("Please ensure the file paths in the 'Configuration' cell are correct.")
    base_graph = None
except Exception as e:
    print(f"\nAn unexpected error occurred during graph loading: {e}")
    base_graph = None


Loading base graph structure...
Dummy load_base_graph_structure called.
Successfully loaded graph: Data(x=[100, 3], edge_index=[2, 200], edge_attr=[200, 3], num_nodes=100)


## 3. Partition Graph using METIS

Run the METIS algorithm on the loaded graph structure to get cluster assignments for each node. This adds the `.part` attribute to the `Data` object.

In [6]:
clustered_data = None
if base_graph is not None:
    print(f"\nPartitioning graph into {NUM_CLUSTERS} clusters...")
    try:
        # The partition_graph function modifies the Data object in-place or returns a new one
        clustered_data = partition_graph(base_graph, NUM_CLUSTERS)
        
        if hasattr(clustered_data, 'part') and clustered_data.part is not None:
            print("Partitioning successful. 'part' attribute added.")
            print(f"Number of nodes: {clustered_data.num_nodes}")
            print(f"Assigned cluster IDs range from {clustered_data.part.min().item()} to {clustered_data.part.max().item()}")
            
            # Basic statistics
            cluster_ids, counts = torch.unique(clustered_data.part, return_counts=True)
            print(f"Number of unique clusters found: {len(cluster_ids)}")
            print(f"Nodes per cluster - Min: {counts.min().item()}, Max: {counts.max().item()}, Avg: {counts.float().mean().item():.2f}")
        else:
            print("Error: Partitioning did not add the 'part' attribute.")
            clustered_data = None # Reset if partitioning failed
            
    except Exception as e:
        print(f"\nAn error occurred during partitioning: {e}")
        clustered_data = None
else:
    print("\nSkipping partitioning because graph loading failed.")



Partitioning graph into 50 clusters...
Dummy partition_graph called for 50 clusters.
Partitioning successful. 'part' attribute added.
Number of nodes: 100
Assigned cluster IDs range from 0 to 48
Number of unique clusters found: 42
Nodes per cluster - Min: 1, Max: 7, Avg: 2.38


## 4. Load Node Coordinates

To visualize the clusters geographically, we need the X and Y coordinates of each node. We load these from the nodes shapefile using geopandas.

In [7]:
node_coords = None
if clustered_data is not None:
    print("\nLoading node coordinates...")
    nodes_shp_path = os.path.join(RAW_DIR, NODES_SHP_FILE)
    try:
        gdf_nodes = gpd.read_file(nodes_shp_path)
        # Ensure the number of geometries matches the number of nodes
        if len(gdf_nodes) != clustered_data.num_nodes:
            print(f"Warning: Node count mismatch! Graph has {clustered_data.num_nodes} nodes, but shapefile has {len(gdf_nodes)} geometries.")
            print("Visualization might be incorrect. Ensure the correct nodes shapefile is used.")
            # Attempt to use coordinates anyway, assuming order matches
            node_coords = np.array([(point.x, point.y) for point in gdf_nodes.geometry[:clustered_data.num_nodes]])
        else:
            # Extract X, Y coordinates
            node_coords = np.array([(point.x, point.y) for point in gdf_nodes.geometry])
            print(f"Successfully loaded coordinates for {len(node_coords)} nodes.")
            
    except Exception as e:
        print(f"\nError loading or processing node coordinates from {nodes_shp_path}: {e}")
        print("Cannot create geographical visualization.")
else:
    print("\nSkipping coordinate loading because partitioning failed.")



Loading node coordinates...

Error loading or processing node coordinates from ./data/raw/your_nodes_file.shp: ./data/raw/your_nodes_file.shp: No such file or directory
Cannot create geographical visualization.


## 5. Visualize the Clusters

Plot the nodes, coloring them according to their assigned cluster ID from the `.part` attribute.

In [9]:
if clustered_data is not None and node_coords is not None:
    print("\nGenerating cluster visualization...")
    
    cluster_assignments = clustered_data.part.cpu().numpy()
    num_actual_clusters = cluster_assignments.max() + 1
    
    plt.figure(figsize=FIGURE_SIZE)
    
    # Create a scatter plot
    scatter = plt.scatter(
        node_coords[:, 0], # X coordinates
        node_coords[:, 1], # Y coordinates
        c=cluster_assignments, # Color by cluster ID
        cmap=COLORMAP,        # Use the chosen colormap
        s=NODE_MARKER_SIZE,   # Marker size
        vmin=0,
        vmax=num_actual_clusters - 1
    )
    
    plt.title(f'Graph Nodes Colored by METIS Cluster Assignment ({num_actual_clusters} Clusters)')
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.axis('equal') # Ensure aspect ratio is equal
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Add a colorbar
    cbar = plt.colorbar(scatter, label='Cluster ID')
    # Optional: Set discrete ticks if number of clusters is small
    # if num_actual_clusters <= 20: 
    #    tick_locs = np.linspace(0, num_actual_clusters - 1, num_actual_clusters)
    #    cbar.set_ticks(tick_locs)
    #    cbar.set_ticklabels(np.arange(num_actual_clusters))
        
    plt.show()
else:
    print("\nSkipping visualization due to previous errors.")



Skipping visualization due to previous errors.
